# BatchTopK lr × batch-size sweep

Analyzes the runs produced by `experiments/sweep_lr_batch.sh`: one BatchTopK SAE per
`(learning-rate, batch-size)` cell of a 2-D grid, at a single width/k. For every
quality metric we render a **heatmap over the (lr × batch) plane** (one per
precision), plus line cuts of each metric vs batch size (one line per lr) so the
batch-size effect is easy to read.

Run dirs are discovered + parsed from disk, so this renders incrementally as cells
finish. The naming convention written by the driver is:

```
saebench_<model>[_fp8][_<run_group>]_lr<lr>_bs<batch>[_g<group_size>]
```

**Batch size is architectural in BatchTopK** (top-k is over the whole batch). The
driver's `GROUP_SIZE` (ghost-batch) knob holds the selection pool fixed so batch
becomes a pure optimization knob; we tag those runs with `_g<size>` and a
`run_group` so raw vs ghost studies don't collide. This notebook keeps them on
separate heatmaps.

In [ ]:
from pathlib import Path
import glob, json, re

import numpy as np
import pandas as pd
import plotly.graph_objects as go

pd.set_option("display.float_format", lambda v: f"{v:.4f}")


def dig(d, *path, default=None):
    """Safely walk nested dicts: dig(metrics, 'sparsity', 'l0')."""
    cur = d
    for p in path:
        if not isinstance(cur, dict) or p not in cur:
            return default
        cur = cur[p]
    return cur


# friendly_name -> (group, key, higher_is_better) within eval_result_metrics.
# (identical registry to notebooks/ksweep_frontier.ipynb so the two agree.)
METRIC_REGISTRY = {
    "core": {
        "L0":            ("sparsity", "l0", None),
        "CE_loss_score": ("model_performance_preservation", "ce_loss_score", True),
        "KL_div_score":  ("model_behavior_preservation", "kl_div_score", True),
        "explained_var": ("reconstruction_quality", "explained_variance", True),
        "frac_alive":    ("misc_metrics", "frac_alive", True),
    },
    "sparse_probing": {
        "probe_top1_acc": ("sae", "sae_top_1_test_accuracy", True),
        "probe_top5_acc": ("sae", "sae_top_5_test_accuracy", True),
        "probe_full_acc": ("sae", "sae_test_accuracy", True),
    },
}
EVAL_TYPES = ["core", "sparse_probing"]
HIGHER_IS_BETTER = {name: hib for reg in METRIC_REGISTRY.values()
                    for name, (_, _, hib) in reg.items()}


def extract_metrics(eval_type, result_json):
    m = result_json.get("eval_result_metrics", {})
    return {name: dig(m, grp, key)
            for name, (grp, key, _) in METRIC_REGISTRY.get(eval_type, {}).items()}

In [ ]:
# --- Config -----------------------------------------------------------------
RESULTS_ROOT = Path("/wekafs/smerrill/efficient_sae/experiments/results")
MODEL = "gemma"          # the model the lr/batch sweep was run on (see driver MODEL=)
WIDTH = 65536
K = 80                    # single sparsity the sweep fixed (driver K=)
MEMBER = f"w{WIDTH}_k{K}"

# saebench_<model>[_fp8][_<run_group>]_lr<lr>_bs<batch>[_g<group_size>]
RUN_RE = re.compile(
    r"^saebench_(?P<model>gemma|pythia)"
    r"(?P<fp8>_fp8)?"
    r"(?:_(?P<run_group>[A-Za-z][A-Za-z0-9]*))?"
    r"_lr(?P<lr>[0-9][0-9.eE+\-]*)"
    r"_bs(?P<bs>\d+)"
    r"(?:_g(?P<gsize>\d+))?$"
)

print("results root:", RESULTS_ROOT)
print("model/member:", MODEL, "/", MEMBER)

In [ ]:
# --- Discover + load every (lr, batch) cell into one wide dataframe ----------
def load_member_final(run_dir: Path, member: str):
    """Metrics dict for the FINAL (latest-step) eval of one member, or {} if none.

    A precise '_<member>_step_' match keeps w65536_k50 from also grabbing k500."""
    eval_dir = run_dir / "saebench_eval" / member / "eval_results"
    if not eval_dir.exists():
        return {}
    best_step, found = -2, {}
    by_step = {}
    for path in glob.glob(str(eval_dir / "*" / "*_eval_results.json")):
        p = Path(path)
        et = p.parent.name
        if et not in EVAL_TYPES or f"_{member}_step_" not in p.name:
            continue
        sm = re.search(r"_step_(-?\d+)", p.name)
        step = int(sm.group(1)) if sm else -1
        data = json.loads(p.read_text())
        by_step.setdefault(step, {}).update(extract_metrics(et, data))
    if not by_step:
        return {}
    return by_step[max(by_step)]


rows = []
for d in sorted(RESULTS_ROOT.glob(f"saebench_{MODEL}*")):
    if not d.is_dir():
        continue
    m = RUN_RE.match(d.name)
    if not m or m.group("model") != MODEL:
        continue
    metrics = load_member_final(d, MEMBER)
    if not metrics:
        continue
    rows.append(dict(
        precision="FP8" if m.group("fp8") else "FP16",
        run_group=m.group("run_group") or "raw",
        lr=float(m.group("lr")),
        bs=int(m.group("bs")),
        gsize=int(m.group("gsize")) if m.group("gsize") else 0,
        run_dir=d.name,
        **metrics,
    ))

df = pd.DataFrame(rows)
assert not df.empty, (
    f"No evaluated lr/batch runs for model='{MODEL}', member='{MEMBER}' under\n  {RESULTS_ROOT}\n"
    "Train+eval first, e.g.:\n"
    "  cd experiments && ./sweep_lr_batch.sh\n"
    "  (or PHASE=eval ./sweep_lr_batch.sh to just (re)evaluate finals)"
)
df = df.sort_values(["run_group", "precision", "lr", "bs"]).reset_index(drop=True)
print(f"loaded {len(df)} (lr,batch) cells | precisions={sorted(df.precision.unique())} "
      f"| run_groups={sorted(df.run_group.unique())}")
print("lrs:", sorted(df.lr.unique()), "| batches:", sorted(df.bs.unique()))
df

## Heatmaps — each metric over the (lr × batch) plane

One heatmap per `(run_group, precision, metric)`. Rows are learning rates, columns
are batch sizes; the cell text is the metric value. For "higher is better" metrics
the brightest cell is the best `(lr, batch)` config; `L0` is just reported (the
achieved sparsity should stay near k).

In [ ]:
HEATMAP_METRICS = ["CE_loss_score", "explained_var", "probe_top1_acc", "L0"]

for run_group, gdf in df.groupby("run_group"):
    for precision, pdf in gdf.groupby("precision"):
        lrs = sorted(pdf.lr.unique())
        batches = sorted(pdf.bs.unique())
        for metric in HEATMAP_METRICS:
            if metric not in pdf.columns or pdf[metric].isna().all():
                continue
            grid = pdf.pivot_table(index="lr", columns="bs", values=metric)
            grid = grid.reindex(index=lrs, columns=batches)
            z = grid.values.astype(float)
            hib = HIGHER_IS_BETTER.get(metric, True)
            fig = go.Figure(go.Heatmap(
                z=z,
                x=[str(b) for b in batches],
                y=[f"{lr:.0e}" for lr in lrs],
                text=[["" if np.isnan(v) else f"{v:.4f}" for v in r] for r in z],
                texttemplate="%{text}", textfont=dict(size=11),
                colorscale="Viridis" if hib is not False else "Viridis_r",
                colorbar=dict(title=metric),
                hovertemplate="lr=%{y}<br>batch=%{x}<br>" + metric + "=%{z:.4f}<extra></extra>",
            ))
            fig.update_layout(
                title=f"[{run_group}] {MODEL} {precision} — {metric} over (lr × batch)  "
                      f"(BatchTopK width {WIDTH}, k={K})",
                xaxis_title="batch size (tokens)", yaxis_title="learning rate",
                width=620, height=440,
            )
            fig.show()

## Batch-size cuts — metric vs batch, one line per lr

The same data sliced the other way. A flat line means the metric is robust to
batch size at that lr; a slope shows BatchTopK's batch-size sensitivity. Compare
the `raw` run_group (batch is architectural) against a `ghost` run_group
(`GROUP_SIZE` fixed) to separate the optimization effect from the architectural one.

In [ ]:
LINE_METRICS = ["CE_loss_score", "explained_var", "probe_top1_acc"]

for run_group, gdf in df.groupby("run_group"):
    for metric in LINE_METRICS:
        if metric not in gdf.columns or gdf[metric].isna().all():
            continue
        fig = go.Figure()
        for (precision, lr), s in gdf.groupby(["precision", "lr"]):
            s = s.dropna(subset=[metric]).sort_values("bs")
            if s.empty:
                continue
            fig.add_trace(go.Scatter(
                x=s["bs"], y=s[metric], mode="lines+markers",
                name=f"{precision} lr={lr:.0e}",
                line=dict(dash="dash" if precision == "FP8" else "solid"),
            ))
        if not fig.data:
            continue
        fig.update_layout(
            title=f"[{run_group}] {MODEL} — {metric} vs batch size  (k={K}, width {WIDTH})",
            xaxis_title="batch size (tokens)", yaxis_title=metric,
            xaxis_type="log", width=820, height=460,
            legend=dict(x=0.99, y=0.01, xanchor="right", yanchor="bottom"),
        )
        fig.show()

## Best config + FP8-vs-FP16 gap

Best `(lr, batch)` per `(run_group, precision)` for each metric, then — where both
precisions trained the *same* cell — the FP8−FP16 difference, so you can confirm the
8-bit GEMMs cost nothing across the lr/batch grid.

In [ ]:
# Best (lr, batch) per metric, per run_group x precision.
best_rows = []
for (run_group, precision), g in df.groupby(["run_group", "precision"]):
    for metric in ["CE_loss_score", "explained_var", "probe_top1_acc"]:
        if metric not in g.columns or g[metric].isna().all():
            continue
        idx = g[metric].idxmax()  # all three are higher-is-better
        r = g.loc[idx]
        best_rows.append(dict(run_group=run_group, precision=precision, metric=metric,
                              best=float(r[metric]), lr=r["lr"], batch=int(r["bs"])))
best = pd.DataFrame(best_rows)
print("Best (lr, batch) per metric:")
best

In [ ]:
# FP8 - FP16 gap on matching (run_group, lr, batch) cells.
metric_cols = [c for c in ["L0", "CE_loss_score", "explained_var", "KL_div_score",
                           "frac_alive", "probe_top1_acc", "probe_top5_acc"]
               if c in df.columns]
gap_rows = []
for (run_group, lr, bs), g in df.groupby(["run_group", "lr", "bs"]):
    fp8 = g[g.precision == "FP8"]
    fp16 = g[g.precision == "FP16"]
    if fp8.empty or fp16.empty:
        continue
    row = {"run_group": run_group, "lr": lr, "batch": int(bs)}
    for c in metric_cols:
        v8, v16 = fp8[c].iloc[0], fp16[c].iloc[0]
        row[f"{c}_fp16"] = v16
        row[f"{c}_fp8"] = v8
        row[f"{c}_diff"] = (v8 - v16) if pd.notna(v8) and pd.notna(v16) else None
    gap_rows.append(row)

gap = pd.DataFrame(gap_rows)
if gap.empty:
    print("No matching FP8 & FP16 cells yet — run both precisions "
          "(PRECISIONS='fp16 fp8' ./sweep_lr_batch.sh) for this table.")
else:
    gap = gap.sort_values(["run_group", "lr", "batch"]).reset_index(drop=True)
gap

## FP8 vs FP16 \u2014 difference over the (lr \u00d7 batch) plane

Needs both precision runs (`PRECISIONS=fp8` on one GPU, `PRECISIONS=fp16` on the
other). For each metric we plot **FP8 \u2212 FP16** per `(lr, batch)` cell on a diverging
scale centered at 0. If FP8's 8-bit GEMMs are lossless, every cell sits near white
(\u22480) across the whole grid \u2014 i.e. the precision choice is independent of the lr/batch
operating point. Strong color in a corner flags an (lr, batch) regime where the
precisions diverge. For `L0` this is just the achieved-sparsity difference.

In [ ]:
DIFF_METRICS = ["CE_loss_score", "explained_var", "probe_top1_acc", "L0"]

for run_group, gdf in df.groupby("run_group"):
    fp8, fp16 = gdf[gdf.precision == "FP8"], gdf[gdf.precision == "FP16"]
    if fp8.empty or fp16.empty:
        print(f"[{run_group}] need both FP8 and FP16 cells for the diff "
              "(run PRECISIONS=fp8 and PRECISIONS=fp16) \u2014 skipping.")
        continue
    lrs = sorted(gdf.lr.unique())
    batches = sorted(gdf.bs.unique())
    for metric in DIFF_METRICS:
        if metric not in gdf.columns:
            continue
        g8 = fp8.pivot_table(index="lr", columns="bs", values=metric).reindex(index=lrs, columns=batches)
        g16 = fp16.pivot_table(index="lr", columns="bs", values=metric).reindex(index=lrs, columns=batches)
        d = (g8 - g16).values.astype(float)
        if np.isnan(d).all():
            continue
        amax = float(np.nanmax(np.abs(d))) or 1e-9
        fig = go.Figure(go.Heatmap(
            z=d,
            x=[str(b) for b in batches],
            y=[f"{lr:.0e}" for lr in lrs],
            text=[["" if np.isnan(v) else f"{v:+.4f}" for v in r] for r in d],
            texttemplate="%{text}", textfont=dict(size=11),
            colorscale="RdBu", zmid=0, zmin=-amax, zmax=amax,
            colorbar=dict(title="FP8\u2212FP16"),
            hovertemplate="lr=%{y}<br>batch=%{x}<br>\u0394=%{z:+.4f}<extra></extra>",
        ))
        fig.update_layout(
            title=f"[{run_group}] {MODEL} \u2014 FP8\u2212FP16 {metric} over (lr \u00d7 batch)  "
                  f"(k={K}, width {WIDTH})",
            xaxis_title="batch size (tokens)", yaxis_title="learning rate",
            width=620, height=440,
        )
        fig.show()